In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime
import random
import time

spark = SparkSession.builder.getOrCreate()

# -----------------------------------------------------
# Stock Configuration
# -----------------------------------------------------
stocks = {
    "MSFT": {"current": 500.0, "min": 450.0, "max": 550.0},
    "NVDA": {"current": 160.0, "min": 130.0, "max": 220.0},
    "TSLA": {"current": 280.0, "min": 220.0, "max": 350.0},
    "AMZN": {"current": 210.0, "min": 180.0, "max": 260.0},
    "GOOG": {"current": 180.0, "min": 150.0, "max": 240.0}
}

# Maintain market trends
market_trends = {
    "MSFT": "BULLISH",
    "NVDA": "SIDEWAYS",
    "TSLA": "BEARISH",
    "AMZN": "SIDEWAYS",
    "GOOG": "BULLISH"
}

# Company IDs
company_ids = {
    "MSFT": "C001",
    "NVDA": "C002",
    "TSLA": "C003",
    "AMZN": "C004",
    "GOOG": "C005"
}


def generate_price(symbol, config):

    current = config["current"]
    lower = config["min"]
    upper = config["max"]

    trend = market_trends[symbol]
    companyId = company_ids[symbol]

    if trend == "BULLISH":
        delta = random.uniform(-1, 4)

    elif trend == "BEARISH":
        delta = random.uniform(-4, 1)

    else:
        delta = random.uniform(-2, 2)

    # occasional spike
    if random.random() < 0.03:
        delta += random.uniform(5, 15)

    # occasional crash
    elif random.random() < 0.03:
        delta -= random.uniform(5, 15)

    new_price = current + delta

    # bounce from boundaries
    if new_price > upper:
        new_price = upper - random.uniform(1, 5)

    if new_price < lower:
        new_price = lower + random.uniform(1, 5)

    config["current"] = round(new_price, 2)

    volume = random.randint(10000, 100000)

    return {
        "EventTime": datetime.utcnow(),
        "CompanyId": companyId,
        "StockSymbol": symbol,
        "Price": round(new_price, 2),
        "Volume": volume,
        "Trend": trend,
        "MarketStatus": "OPEN"
    }

schema = StructType([
    StructField("EventTime", TimestampType(), True),
    StructField("CompanyId", StringType(), True),
    StructField("StockSymbol", StringType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Volume", LongType(), True),
    StructField("Trend", StringType(), True),
    StructField("MarketStatus", StringType(), True)
])

eventhouse_cluster_uri = "<event-house-cluster-uri>"
eventhouse_database = "<database-name>"
access_token=mssparkutils.credentials.getToken(eventhouse_cluster_uri)

while True:

    events = []

    for symbol, config in stocks.items():
        events.append(generate_price(symbol, config))

    df = spark.createDataFrame(events, schema)

    display(df)

    df.write.format("com.microsoft.kusto.spark.synapse.datasource").option("kustoCluster",eventhouse_cluster_uri).option("kustoDatabase",eventhouse_database).option("kustoTable", 'StockPriceEvents').option("accessToken", access_token).option("tableCreateOptions", "CreateIfNotExist").mode("Append").save()  

    print(f"Written {len(events)} records at {datetime.utcnow()}")

    time.sleep(30)

StatementMeta(, b50f65f0-53e2-4dd1-acca-d91c19d38758, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e79b7851-13ea-422b-9a53-82774b80a047)

Written 5 records at 2026-07-07 07:18:01.319462


SynapseWidget(Synapse.DataFrame, b59e4358-d2ac-45d8-b0fb-b66a8c70c1fd)

Written 5 records at 2026-07-07 07:19:43.298610


SynapseWidget(Synapse.DataFrame, 88ecce95-222b-49f2-b8ee-e8ab48826188)

... (75 outputs left out)

SynapseWidget(Synapse.DataFrame, ed22bbdd-570a-45a4-bc6d-aaa9670d377c)

Written 5 records at 2026-07-07 08:23:29.793560


SynapseWidget(Synapse.DataFrame, 244591c4-02c7-43c3-ab85-2fc5287304fe)

Py4JJavaError: An error occurred while calling o135372.save.
: com.microsoft.azure.kusto.data.exceptions.DataServiceException: Http StatusCode='401', ActivityId='288afe8c-7d17-4c3c-8384-6247b18d5577'
	at com.microsoft.azure.kusto.data.BaseClient.createExceptionFromResponse(BaseClient.java:167)
	at com.microsoft.azure.kusto.data.BaseClient.lambda$null$0(BaseClient.java:60)
	at kusto_connector_shaded.reactor.core.publisher.FluxMap$MapSubscriber.onNext(FluxMap.java:106)
	at kusto_connector_shaded.reactor.core.publisher.FluxSwitchIfEmpty$SwitchIfEmptySubscriber.onNext(FluxSwitchIfEmpty.java:74)
	at kusto_connector_shaded.reactor.core.publisher.FluxDoFinally$DoFinallySubscriber.onNext(FluxDoFinally.java:113)
	at kusto_connector_shaded.reactor.core.publisher.MonoIgnoreThen$ThenIgnoreMain.complete(MonoIgnoreThen.java:292)
	at kusto_connector_shaded.reactor.core.publisher.MonoIgnoreThen$ThenIgnoreMain.onNext(MonoIgnoreThen.java:187)
	at kusto_connector_shaded.reactor.core.publisher.MonoIgnoreT